# ⚡ Google Colab Tesla T4 GPU &rarr; Mac Terminal Tunnel
### **Connect your local macOS Terminal directly into Google Colab's Free NVIDIA Tesla T4 GPU**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/notebooks/intro.ipynb)

**Lead Architect:** Ayush Pandey &nbsp;|&nbsp; **Collaborator:** Bhaskar Pandey (`bhaskarauthor@gmail.com`)  
**Workspace:** `MyDrive/Databento_Gold_Futures_Research`  

---

## 📌 Instructions
1. In the top menu, make sure GPU is active: **Runtime &rarr; Change runtime type &rarr; T4 GPU &rarr; Save**.
2. Run **Cell 1** to mount Google Drive and switch into the project directory.
3. Run **Cell 2 (Instant Tmate SSH)** &mdash; it generates an `ssh ...` command with **zero token and zero signup**.
4. Copy that command and paste it into your local Mac terminal.
5. **You are now working inside Colab with full GPU access directly from your Mac terminal!**

## 🔌 Step 1: Mount Google Drive & Set Project Directory

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    print("[*] Mounting Google Drive...")
    drive.mount('/content/drive')
    
    repo_candidates = [
        Path('/content/drive/MyDrive/Databento_Gold_Futures_Research'),
        Path('/content/drive/My Drive/Databento_Gold_Futures_Research')
    ]
    repo_root = None
    for p in repo_candidates:
        if p.exists():
            repo_root = p
            break
            
    if repo_root:
        os.chdir(str(repo_root))
        print(f"[✓] Working directory set to: {os.getcwd()}")
    else:
        print("[!] Project folder not found in default Google Drive path.")
else:
    print(f"[✓] Local directory: {os.getcwd()}")

## 🚀 Step 2: Launch Instant Terminal Tunnel (Zero-Signup Tmate SSH)
This installs `tmate`, launches a background session, and gives you a single `ssh ...` command to paste in your Mac terminal.

In [ ]:
# Install and start tmate tunnel
!apt-get update -qq && apt-get install -y -qq tmate openssh-client

import subprocess
import time

# Start tmate session
subprocess.run(["tmate", "-S", "/tmp/tmate.sock", "new-session", "-d"])
subprocess.run(["tmate", "-S", "/tmp/tmate.sock", "wait", "tmate-ready"])

# Fetch SSH connection string
ssh_cmd = subprocess.check_output(["tmate", "-S", "/tmp/tmate.sock", "display", "-p", "#{tmate_ssh}"]).decode().strip()

print("\n" + "=" * 75)
print(" 🚀 GPU ATTACHED! COPY & PASTE THIS COMMAND INTO YOUR MAC TERMINAL:")
print("=" * 75)
print(f"\n  {ssh_cmd}\n")
print("=" * 75)
print("[*] What you can run from your Mac terminal once connected:")
print("    nvidia-smi                        # Check Tesla T4 GPU (16 GB VRAM)")
print("    cd /content/drive/MyDrive/Databento_Gold_Futures_Research")
print("    python3 scripts/train_colab_t4_gpu.py")
print("    python3 scripts/backtest_colab_t4_gpu.py")
print("    python3 manage.py pnl")
print("=" * 75)

## 🌐 Step 3 (Alternative A): Colab-XTerm (In-Browser Terminal Tab)
If you prefer an in-browser bash terminal without leaving your web browser, run the cell below:

In [ ]:
!pip install -q colab-xterm
%load_ext colabxterm
%xterm

## 🔒 Step 3 (Alternative B): Ngrok SSH Tunnel
If you have a free [ngrok.com](https://ngrok.com) account, you can use ngrok TCP tunnel:

In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok, conf

token = input("Enter your Ngrok Authtoken (from dashboard.ngrok.com): ").strip()
if token:
    conf.get_default().auth_token = token
    
    # Set root password and configure sshd
    !apt-get install -qq -o=Dpkg::Use-Pty=0 openssh-server > /dev/null
    !mkdir -p /var/run/sshd
    !echo "PermitRootLogin yes" >> /etc/ssh/sshd_config
    !echo "PasswordAuthentication yes" >> /etc/ssh/sshd_config
    !echo "root:cme_gold_2026" | chpasswd
    !/usr/sbin/sshd
    
    # Connect tunnel
    ssh_tunnel = ngrok.connect(22, "tcp")
    host = ssh_tunnel.public_url.replace("tcp://", "").split(":")[0]
    port = ssh_tunnel.public_url.split(":")[-1]
    
    print("\n" + "=" * 75)
    print(" 🚀 NGROK SSH READY! RUN IN YOUR MAC TERMINAL:")
    print("=" * 75)
    print(f"\n  ssh root@{host} -p {port}\n")
    print("  Password: cme_gold_2026")
    print("=" * 75)